# Flood Mapping with EODC Dask Gateway

Welcome to this workshop!

Here we intend to show the steps to create Floods Map with Sentinel-1 radar images. We replicate in this package the work of Bauer-Marschallinger et al. (2022) on the TU Wien Bayesian-based flood mapping algorithm. 
The computation is carried out remotely via EODC Dask Gateway and the data are accessed vis STAC with odc-stac.



The notebook is organized as:

1. [Set Connection to EODC Dask Gateway](#set-connection-to-eodc-dask-gateway)
2. [Use case: Northern Germany Flood](#2-use-case-northern-germany-flood)
3. [Getting the data: EODC STAC Catalogue](#3-getting-the-data-eodc-stac-catalog)

    3.1 [Microwave Backscatter Measurements](#31-microwave-backscatter-measurements)

    3.2 [Harmonic Parameters](#32-harmonic-parameters)  

    3.3 [Local Incidence Angles (LAI)](#33-local-incidence-angles-lai)

    3.4 [GFM Water Mask](#34-gfm-water-mask)
4. [Fuse Cube](#4-fuse-cube)
5. [Likelihoods](#5-likelihoods) 

    5.1 [Water](#51-water)

    5.2 [Land](#52-land)
6. [Flood Mapping](#6-flood-mapping)
7. [Post Processing](#7-postprocessing)  

    7.1 [Removal of Speckles](#71-removal-of-speckles) 

8. [Results](#8-results)   
    

## 1. Set connection to EODC Dask Gateway
Autentication is required through an username and password that should be requested to EODC.

Import of required packages

In [ ]:
# Import of required packages 

import os
os.environ["AWS_NO_SIGN_REQUEST"] = "YES"
import hvplot.xarray  # noqa
from dask_flood_mapper import flood
from eodc import settings
from eodc.dask import EODCDaskGateway
from rich.console import Console
from rich.prompt import Prompt
import os
import numpy as np
import pystac_client
import xarray as xr
from dask.distributed import Client, wait
from odc import stac as odc_stac

settings.DASK_URL = "http://dask.services.eodc.eu"
settings.DASK_URL_TCP = "tcp://dask.services.eodc.eu:10000/"

Accessing the cluster through login

In [ ]:
console = Console()
your_username = Prompt.ask(prompt="Enter your Username")
gateway = EODCDaskGateway(username=your_username)

Definition of cluster configuration

In [ ]:
# Cluster configuration
 
cluster_options = gateway.cluster_options()
cluster_options.image = "ghcr.io/eodcgmbh/cluster_image:2025.4.1"
cluster_options.worker_cores = 8
cluster_options.worker_memory = 16
cluster = gateway.new_cluster(cluster_options)
cluster.scale(5) # Or adaptative scale: cluster.adapt(minimum=3, maximum=8)
client = cluster.get_client()
cluster

Accessing Dask dashboard

In [ ]:
cluster.dashboard_link 

#### Cube and chunk Definitions


Setting up the coordinate reference system, resolution and size of chunks for Dask optimize the performance.


In [6]:
# Coordinate Reference System - World Geodetic System 1984 (WGS84) in this case
crs = "EPSG:4326"

# Resolution
res = 0.00018  

# Size of data chunks
chunks = {"time": 1, "latitude": 1300, "longitude": 1300}

## 2. Use case: Northern Germany Flood

Storm Babet hit the Denmark and Northern coast at the 20th of October 2023 [Wikipedia](https://en.wikipedia.org/wiki/Storm_Babet). 

An area around Zingst at the Baltic coast of Northern Germany is selected as the study area.


In [7]:
# Time range of the event
time_range = "2023-10-11/2023-10-25"

# Definition of the area trough latitude and longitude
minlon, maxlon = 12.3, 13.1
minlat, maxlat = 54.3, 54.6
bounding_box = [minlon, minlat, maxlon, maxlat]

## 3. Getting the data: EODC STAC Catalog

The data is obtained trought the EODC STAC Catalog. The connection to it is made via `pystac_client`.

We will get and process the following data:

* 3.1 [Microwave Backscatter Measuaments](#31-microwave-backscatter-measurements)
* 3.2 [Harmonic Parameters](#32-harmonic-parameters)
* 3.3 [Local Incidence Angles](#33-local-incidence-angles-lai) 
* 3.4 [GFM World Cover](#34-gfm-water-mask)


In [8]:
# Create the catalog object from EODC
eodc_catalog = pystac_client.Client.open("https://stac.eodc.eu/api/v1")

# Get all collections
collections = eodc_catalog.get_collections()

### 3.1 Microwave Backscatter Measurements

The characteristics of backscattering over land and water differ considerably. With this knowledge we can detect whenever a pixel with a predominant land like signature changes to a water like signature in the event of flooding.

<figure style="text-align: center;">
  <img src="https://www.gsi.ie/images/images/SAR_mapping_land_water.jpg" alt="SAR Map" width="400">
  <figcaption>*Schematic backscattering over land and water. Image from [Geological Survey Ireland](https://www.gsi.ie/images/images/SAR_mapping_land_water.jpg)*</figcaption>
</figure>

Discover the items of Sentinel-1 microwave backscatter ($\sigma_0$ [1]) at a 20 meter resolution 

In [9]:
# Search the specific collection at the edoc catalog
search = eodc_catalog.search(
    collections="SENTINEL1_SIG0_20M",
    bbox=bounding_box,
    datetime=time_range,
)

# Get the items from this collection
items_sig0 = search.item_collection()

Helper functions are defined to extract specific information from the items

In [10]:
# Helper functions

""" Functions  created to help extract some required information as:
* Orbit state
* Relative orbit number
* scaling factor
* nodata values """

# Function to extract orbit state and relative orbit number
def extract_orbit_names(items):
    return np.array(
        [
            items[i].properties["sat:orbit_state"][0].upper()
            + str(items[i].properties["sat:relative_orbit"])
            for i in range(len(items))
        ]
    )

# apply post process for each dataset in the cube
def post_process_eodc_cube(dc: xr.Dataset, items, bands):
    if not isinstance(bands, tuple):
        bands = tuple([bands])
    for i in bands:
        dc[i] = post_process_eodc_cube_(
            dc[i], items, i
        )  
    return dc


def post_process_eodc_cube_(dc: xr.Dataset, items, band):

    # obtain the scaling factor
    scale = items[0].assets[band].extra_fields.get("raster:bands")[0]["scale"]
    
    #obtain the nodata values
    nodata = items[0].assets[band].extra_fields.get("raster:bands")[0]["nodata"]

    return dc.where(dc != nodata) / scale


We now load the VV polarization of the discover items with `odc-stac` with the previous defined projection and resolution. 

The data is at this point only lazily loaded.


In [11]:
bands = "VV"
sig0_dc = odc_stac.load(
    items_sig0,
    bands=bands,
    crs=crs,
    chunks=chunks,
    resolution=res,
    bbox=bounding_box,
    resampling="bilinear",
    groupby=None,
)

The data is then prepared for the next steps by: fill the no data values with `np.nan` values, extract the orbit names, remove duplicated time dimension.

In [ ]:
# Data preparation 

sig0_dc = (
    post_process_eodc_cube(sig0_dc, items_sig0, bands)
    .rename_vars({"VV": "sig0"})
    .assign_coords(orbit=("time", extract_orbit_names(items_sig0)))
    .dropna(dim="time", how="all")
    .sortby("time")
)
__, indices = np.unique(sig0_dc.time, return_index=True)
indices.sort()
orbit_sig0 = sig0_dc.orbit[indices].data
sig0_dc = sig0_dc.groupby("time").mean(skipna=True)
sig0_dc = sig0_dc.assign_coords(orbit=("time", orbit_sig0))
sig0_dc = sig0_dc.persist()

### 3.2 Harmonic Parameters

The so-called likelihoods of $P(\sigma^0|flood)$ and $P(\sigma^0|nonflood)$ can be calculated from past backscattering information. 

To be able to do this, we load the harmonic parameters so we can model the expected variations in land back scattering based on seasonal changes in vegetation. The procedure is similar to the backscattering routine.



Discover Harmonic Paramters items.

In [13]:
# Search the specific collection at the EODC Catalog
search = eodc_catalog.search(collections="SENTINEL1_HPAR", bbox=bounding_box)

# Get the items
items_hpar = search.item_collection()

Load the data as a lazy object.


In [14]:
# Define the bands
bands = ("C1", "C2", "C3", "M0", "S1", "S2", "S3", "STD")

# Load the data
hpar_dc = odc_stac.load(
    items_hpar,
    bands=bands,
    crs=crs,
    chunks=chunks,
    resolution=res,
    bbox=bounding_box,
    groupby=None,
)

# Data preparation
hpar_dc = post_process_eodc_cube(hpar_dc, items_hpar, bands).rename({"time": "orbit"})
hpar_dc["orbit"] = extract_orbit_names(items_hpar)
hpar_dc = hpar_dc.groupby("orbit").mean(skipna=True)
hpar_dc = hpar_dc.sel(orbit=orbit_sig0)
hpar_dc = hpar_dc.persist()


### 3.3 Local Incidence Angles (LAI)

Local incidence angles of measured microwave backscattering is as well important for calculating reference backscatter values, but now for water bodies. The procedure is much the same as for the harmonic parameters.


In [15]:
# Search the specific collection at the edoc catalog
search = eodc_catalog.search(collections="SENTINEL1_MPLIA", bbox=bounding_box)

# Get the items
items_plia = search.item_collection()

Load the lazy object and preprocess.


In [16]:
# Define the band
bands = "MPLIA"

# Load the data
plia_dc = odc_stac.load(
    items_plia,
    bands=bands,
    crs=crs,
    chunks=chunks,
    resolution=res,
    bbox=bounding_box,
    groupby=None,
)

# Data preparation
plia_dc = post_process_eodc_cube(plia_dc, items_plia, bands).rename({"time": "orbit"})
plia_dc["orbit"] = extract_orbit_names(items_plia)
plia_dc = plia_dc.groupby("orbit").mean(skipna=True)
plia_dc = plia_dc.sel(orbit=orbit_sig0)
plia_dc = plia_dc.persist()

### 3.4 GFM Water Mask 

For flood mapping we are only interested in microwave backscattering over what used to be land, as such, we need a way to mask water bodies. For this we use the GFM Water Mask from  Cover data from the EODC catalog.


Similarly, we discover the required items and load the data.

In [17]:
# Search the specific collection at the EODC catalog
search_GFM = eodc_catalog.search(
    collections="GFM",
    bbox=bounding_box,
    datetime="2023-10-11T05:33:43.000000000", # choose one time stamp for simplification an avoid errors
)

# Get the items
items_GFM = search_GFM.item_collection()

# Load the data
GFM_dc= (odc_stac.load(
    items_GFM, 
    bbox=bounding_box,   
    crs=crs,   
    bands=["reference_water_mask"],   
    resolution=res   
    )
    .squeeze("time")
    .drop_vars("time")
    .rename_vars({"reference_water_mask": "wcover"}))

## 4. Fuse cube

At this point we have created 4 datacubes: Microwave backscatter, Harmonic Parameters, Local Incidence Angles and Water mask.

Now we need to fuse all the four and filter HAND value of above zero. Additionally, we can drop the orbit coordinates, as well as time slices that contain no land backscattering data.


In [ ]:
# Merge the 4 data cubes
flood_dc = xr.merge([sig0_dc, plia_dc, hpar_dc, GFM_dc])

# Replace 255 to NAN in the water mask
flood_dc = flood_dc.where(flood_dc.wcover != 255)

# Reset orbit index
flood_dc = (
    flood_dc.reset_index("orbit", drop=True))

# Rename orbit
flood_dc = flood_dc.rename({"orbit": "time"})

# Drop empty elements 
flood_dc = flood_dc.dropna(dim="time", how="all", subset=["sig0"])

Visualize the created cube

In [ ]:
flood_dc

## 5. Likelihoods

Now we are ready to calculate the likelihoods of micorwave backscattering given flooding (or non flooding).


### 5.1 Water

We start with water which is the simplest calculation, where the hard coded values are coefficients of a regression model fitted to global water backscattering values.  


In [23]:
# Function to calculate water likelihood with hard-coded values
def calc_water_likelihood(dc):
    return dc.MPLIA * -0.394181 + -4.142015

# Applying the function to the datacube
flood_dc["wbsc"] = calc_water_likelihood(flood_dc)

### 5.2 Land

For land backscattering, we construct the harmonic model from the parameters contained in the fused data cube. 

By doing so, we obtain a reference land backscattering value to compare with the actual observed sigma naught values.

In [25]:
def harmonic_expected_backscatter(dc):
    w = np.pi * 2 / 365

    t = dc.time.dt.dayofyear
    wt = w * t

    M0 = dc.M0
    S1 = dc.S1
    S2 = dc.S2
    S3 = dc.S3
    C1 = dc.C1
    C2 = dc.C2
    C3 = dc.C3
    hm_c1 = (M0 + S1 * np.sin(wt)) + (C1 * np.cos(wt))
    hm_c2 = (hm_c1 + S2 * np.sin(2 * wt)) + C2 * np.cos(2 * wt)
    hm_c3 = (hm_c2 + S3 * np.sin(3 * wt)) + C3 * np.cos(3 * wt)
    return hm_c3

flood_dc["hbsc"] = harmonic_expected_backscatter(flood_dc)

## 6. Flood mapping

Having calculated the likelihoods, we can now move on to calculate the probability of floof given a pixel's $\sigma^0$. 
For that we use Bayesian statistics, where we first assume the values of the flood/not flood probability as 50%/50%, those are called *priors*. Afterwards we update those using the likelihood we just calculated to uptade those values, which are nowcaled *posteriors*.







The following code block shows how we calculate the priors which allow use to predict whether it is likely if a land pixel became flooded.


In [27]:
def bayesian_flood_decision(dc):
    nf_std = 2.754041
    sig0 = dc.sig0
    std = dc.STD
    wbsc = dc.wbsc
    hbsc = dc.hbsc

    f_prob = (1.0 / (std * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * (((sig0 - wbsc) / nf_std) ** 2)
    )
    nf_prob = (1.0 / (nf_std * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * (((sig0 - hbsc) / nf_std) ** 2)
    )

    evidence = (nf_prob * 0.5) + (f_prob * 0.5)
    nf_post_prob = (nf_prob * 0.5) / evidence
    f_post_prob = (f_prob * 0.5) / evidence
    decision = xr.where(
        np.isnan(f_post_prob) | np.isnan(nf_post_prob),
        np.nan,
        np.greater(f_post_prob, nf_post_prob),
    )
    return nf_post_prob, f_post_prob, decision

flood_dc[["nf_post_prob", "f_post_prob", "decision"]] = bayesian_flood_decision(
    flood_dc
)

## 7. Postprocessing

We continue by improving our flood map by filtering out observations that we expect to have low sensitivity to flooding based on a predefined set of criteria.

These criteria include:
* Masking of Exceeding Incidence Angles
* Identification of Conflicting Distributions
* Removal of Measurement Outliers
* Denial of High Uncertainty on Decision



In [29]:
def post_processing(dc):
    dc = dc * np.logical_and(dc.MPLIA >= 27, dc.MPLIA <= 48)
    dc = dc * (dc.hbsc > (dc.wbsc + 0.5 * 2.754041))
    land_bsc_lower = dc.hbsc - 3 * dc.STD
    land_bsc_upper = dc.hbsc + 3 * dc.STD
    water_bsc_upper = dc.wbsc + 3 * 2.754041
    mask_land_outliers = np.logical_and(
        dc.sig0 > land_bsc_lower, dc.sig0 < land_bsc_upper
    )
    mask_water_outliers = dc.sig0 < water_bsc_upper
    dc = dc * (mask_land_outliers | mask_water_outliers)
    return (dc * (dc.f_post_prob > 0.8)).decision

flood_output = post_processing(flood_dc)

### 7.1 Removal of Speckles

Speckles are areas of one or a few pixels, and which are likely the result of the diversity of scattering surfaces at a sub-pixel level. In this approach it is argued that small, solitary flood surfaces are unlikely. Hence speckles are removed by applying a smoothing filter which consists of a rolling window median along the x and y-axis simultaneously.


In [ ]:
flood_output = (
    flood_output.rolling({"longitude": 5, "latitude": 5}, center=True)
    .median(skipna=True)
    .persist()
)
wait(flood_output)

## 8. Results

In the following graphic we superimpose the data on a map and we can move the slider to see which areas become flooded over time.

In [ ]:
flood_output.hvplot.image(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    tiles=True,
    project=True,
    cmap=["rgba(0, 0, 1, 0.1)", "darkred"],
    cticks=[(0, "non-flood"), (1, "flood")],
    frame_height=400,
)

In [33]:
# shoutdown the cluter
cluster.close(shutdown=True)